# Silver — ERP Product Categories
Category / subcategory lookup for products.

`bronze.erp_px_cat_g1v2` → `silver.erp_product_category`

## Init

In [ ]:
import os, sys
import pyspark.sql.functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import DateType

# Make src/ importable from wherever this notebook runs (git folder, bundle, VS Code sync)
root = os.getcwd()
while not os.path.isdir(os.path.join(root, "src")) and root != "/":
    root = os.path.dirname(root)
sys.path.insert(0, os.path.join(root, "src"))

from lakehouse.transforms import trim_strings, normalize, rename, yyyymmdd_to_date

CATALOG = "workspace"

## Read bronze table

In [ ]:
df = spark.table(f"{CATALOG}.bronze.erp_px_cat_g1v2")

## Transformations

### Trim all string columns

In [ ]:
df = trim_strings(df)

### Maintenance flag to boolean

In [ ]:
df = df.withColumn(
    "maintenance",
    F.when(F.upper(col("maintenance")) == "YES", F.lit(True))
     .when(F.upper(col("maintenance")) == "NO", F.lit(False))
     .otherwise(None)
)

### Rename to business-friendly names

In [ ]:
RENAME_MAP = {
    "id": "category_id",
    "cat": "category",
    "subcat": "subcategory",
    "maintenance": "maintenance_flag"
}
df = rename(df, RENAME_MAP)

## Sanity check

In [ ]:
df.limit(10).display()

## Write silver table

In [ ]:
df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(f"{CATALOG}.silver.erp_product_category")

In [ ]:
%sql
SELECT * FROM workspace.silver.erp_product_category LIMIT 10;